# Integrative trajectory and gene regulatory network inference for cancer immunotherapy single-cell RNA-seq

This notebook provides a step-by-step workflow to study T-cell state transitions in cancer immunotherapy using single-cell RNA-seq data. It mirrors the logic of the accompanying Springer Nature protocol and is written to be easy to follow through the trajectory, branch, and pathway analysis steps.

## What this notebook covers
- loading the squamous cell carcinoma dataset `GSE123813`
- selecting a biologically coherent T-cell compartment
- preprocessing, batch correction, clustering, and UMAP inspection
- trajectory inference with PAGA, Monocle 3, and Slingshot
- interpretation of dynamic transcription factor and pathway programs along pseudotime
- an advanced handoff to pySCENIC-based gene regulatory network analysis

## Audience and scope
- The sections up to pathway analysis are written for users who are comfortable with Python notebooks but may be new to trajectory inference.
- The pySCENIC and GRN sections assume more experience with external files, command-line tools, and HPC environments.

## Inputs
- a raw counts matrix with genes as rows and cells as columns
- a cell metadata table containing sample information and cell annotations
- for the GRN section, pySCENIC outputs such as adjacency tables and regulon annotations

## Expected outputs
- a quality-controlled and annotated `AnnData` object
- embeddings and trajectory visualizations
- pseudotime-associated TF and pathway activity summaries
- an interpretable starting point for downstream GRN analysis

## Practical notes
- The downsampled example can run on a laptop, but the full dataset and pySCENIC steps require substantially more memory.
- Annotation quality and root selection strongly affect the biological interpretation of pseudotime.

# Table of Contents

- [Notebook configuration](#notebook-configuration)
- [4.1 Data acquisition and preparation](#41-data-acquisition-and-preparation)
- [4.2 Selection of cells of interest](#42-selection-of-cells-of-interest)
- [4.3 Quality control and normalization](#43-quality-control-and-normalization)
- [4.4 Batch and patient-effect handling, clustering, and UMAP visualization](#44-batch-and-patient-effect-handling-clustering-and-umap-visualization)
- [4.5 Trajectory inference analysis](#45-trajectory-inference-analysis)
- [4.5.1 Root selection criteria](#451-root-selection-criteria)
- [4.5.2 PAGA analysis](#452-paga-analysis)
- [4.5.3 Monocle 3 analysis](#453-monocle-3-analysis)
- [4.5.4 Slingshot analysis](#454-slingshot-analysis)
- [4.5.5 Trajectory validation](#455-trajectory-validation)
- [4.6 Transcription factor activity analysis along pseudotime](#46-transcription-factor-activity-analysis-along-pseudotime)
- [4.7 Genes varying along the principal graph](#47-genes-varying-along-the-principal-graph)
- [4.8 Pathway enrichment analysis](#48-pathway-enrichment-analysis)
- [4.9 Gene regulatory network inference with pySCENIC](#49-gene-regulatory-network-inference-with-pyscenic)
- [4.10 Differential regulon activity across branches](#410-differential-regulon-activity-across-branches)
- [4.11 TF-TF network construction and interpretation](#411-tf-tf-network-construction-and-interpretation)

In [1]:
## Import required packages
import warnings
warnings.filterwarnings("ignore")

from IPython.display import display

import decoupler as dc
import harmonypy as hm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
from py_monocle import differential_expression_genes, learn_graph, order_cells
from pyslingshot import Slingshot
from scipy import sparse

## Notebook configuration

Edit the parameters in the next cell before running the workflow. The defaults reproduce a CD4, pre-treatment example and optionally downsample the data for a faster demonstration run. For a publication analysis, use the full filtered dataset whenever possible and record any deviations from the default settings.

In [2]:
# Core input settings
DATA_DIR = "data/GSE123813"
CELLTYPE_KEY = "celltype"
PATIENT_KEY = "patient"
TREATMENT_KEY = "treatment"

# Example biological question: CD4 T-cell trajectories before treatment
CELLTYPES_OF_INTEREST = ["Naive", "Tfh", "Th17", "Treg"]
TREATMENT_TO_KEEP = "pre"  # set to None to keep all treatments
USE_DOWNSAMPLED_EXAMPLE = True
DOWNSAMPLE_N_CELLS = 5000
RANDOM_STATE = 42

# Quality-control and embedding settings
MIN_GENES_PER_CELL = 200
MIN_CELLS_PER_GENE = 3
N_GENES_LOWER = 1000
N_GENES_UPPER = 4000
MAX_MT_PCT = 8
MAX_TOTAL_COUNTS = 12000
N_TOP_HVGS = 2000
N_NEIGHBORS = 35
N_PCS = 16
LEIDEN_RESOLUTION = 0.3
UMAP_MIN_DIST = 0.7

# Trajectory settings
ROOT_CELLTYPE = "Naive"
MONOCLE_ROOT_INDEX = 0
SLINGSHOT_START_NODE = 0  # update after inspecting cluster ordering if needed

# 4.1 Data acquisition and preparation

This protocol uses the publicly available single-cell RNA-seq dataset `GSE123813`, which contains tumor-infiltrating lymphocytes from squamous cell carcinoma patients sampled before and after anti-PD-1 treatment. The dataset includes a raw count matrix and cell-level metadata with author-provided annotations.

At this stage, the goal is to create a clean `AnnData` object and confirm that the counts matrix, barcodes, and metadata are aligned before any biological filtering or trajectory analysis.

## 4.1.1 Data download

Create a directory for the example dataset and download the metadata and count matrix from GEO. If you are applying the workflow to your own data, replace this step with the paths to your own count matrix and metadata files.

In [3]:
# Download the example dataset used throughout the protocol.
!mkdir -p data/GSE123813

!wget -q -P data/GSE123813 \
https://ftp.ncbi.nlm.nih.gov/geo/series/GSE123nnn/GSE123813/suppl/GSE123813_scc_metadata.txt.gz

!wget -q -P data/GSE123813 \
https://ftp.ncbi.nlm.nih.gov/geo/series/GSE123nnn/GSE123813/suppl/GSE123813_scc_scRNA_counts.txt.gz

## 4.1.2 Data loading

Load the count matrix and metadata into Python and construct an `AnnData` object. The GEO matrix is stored with genes as rows and cells as columns, so we transpose it to match the `AnnData` convention of cells as rows and genes as columns.

In [4]:
metadata = pd.read_csv(f"{DATA_DIR}/GSE123813_scc_metadata.txt.gz", sep="\t")
counts = pd.read_csv(f"{DATA_DIR}/GSE123813_scc_scRNA_counts.txt.gz", sep="\t", index_col=0)

adata = sc.AnnData(counts.T)

# Align metadata to the cell barcodes in the count matrix.
assert adata.obs_names.isin(metadata.iloc[:, 0]).all(), "Some cell barcodes are missing from the metadata table."
adata.obs = metadata.set_index(metadata.columns[0]).loc[adata.obs_names]

# Reuse the author-provided cluster annotation as a working cell-type label when needed.
if CELLTYPE_KEY not in adata.obs.columns and "cluster" in adata.obs.columns:
    adata.obs[CELLTYPE_KEY] = adata.obs["cluster"]

## 4.1.3 Data inspection

Before starting downstream analysis, inspect the dimensions of the dataset, the available metadata fields, and the distribution of annotated cell states. This early checkpoint is useful for catching mismatched identifiers, formatting problems, or missing biological groups before you spend time on quality control and trajectory inference.

In [5]:
print(adata)
print(f"Number of cells: {adata.n_obs}")
print(f"Number of genes: {adata.n_vars}")
print("\nMetadata columns:")
print(list(adata.obs.columns))

columns_to_show = [col for col in [CELLTYPE_KEY, TREATMENT_KEY, PATIENT_KEY] if col in adata.obs.columns]
display(adata.obs[columns_to_show].head())

adata.obs[CELLTYPE_KEY].value_counts()

AnnData object with n_obs × n_vars = 26016 × 18347
    obs: 'patient', 'treatment', 'cluster', 'UMAP1', 'UMAP2', 'celltype'
Number of cells: 26016
Number of genes: 18347

Metadata columns:
['patient', 'treatment', 'cluster', 'UMAP1', 'UMAP2', 'celltype']


,celltype,treatment,patient
scc.su010.pre.rep1_AAACCTGCATGTTCCC,CD8_naive,pre,su010
scc.su010.pre.rep1_AAACGGGAGGACAGAA,CD8_naive,pre,su010
scc.su010.pre.rep1_AAACGGGCAAGTCTAC,Th17,pre,su010
scc.su010.pre.rep1_AAACGGGGTAGTACCT,CD8_naive,pre,su010
scc.su010.pre.rep1_AAACGGGGTGATAAAC,CD8_naive,pre,su010


celltype
CD8_mem       5333
CD8_ex        4149
CD8_naive     3159
Treg          3087
Tfh           3085
Naive         2501
Th17          1783
CD8_act       1272
CD8_eff       1032
CD8_ex_act     615
Name: count, dtype: int64

# 4.2 Selection of cells of interest

Trajectory inference should be performed on a biologically coherent subset of cells that plausibly lie on a shared continuum. Mixing unrelated immune lineages can obscure true lineage relationships and create artificial transitions.

Typical decisions at this stage include:
- analysing CD4 T cells only
- analysing CD8 T cells only
- excluding unrelated immune or stromal populations
- optionally removing cycling cells if they dominate the manifold
- deciding whether pre-treatment and post-treatment samples should be analysed jointly or separately

In this example, we focus on a CD4 compartment and keep the treatment setting explicit so that pre-treatment and post-treatment trajectories can be compared in a principled way.

**Critical step.** Decide early whether you want a single trajectory for all selected T cells or separate analyses for CD4, CD8, and treatment-specific compartments.

If populations differ strongly between conditions, analyse them separately. If populations overlap and you want a shared manifold, analyse them jointly. In this dataset, treatment can substantially shift cell-state composition, so separate analyses are often easier to interpret.

If you do not have trusted cell-type annotations, this is the right point to introduce an annotation step with marker genes or a reference-based classifier such as CellTypist before continuing.

In [ ]:
# Keep only the biologically relevant compartment for the trajectory analysis.
adata = adata[adata.obs[CELLTYPE_KEY].isin(CELLTYPES_OF_INTEREST)].copy()

if TREATMENT_TO_KEEP is not None:
    adata = adata[adata.obs[TREATMENT_KEY] == TREATMENT_TO_KEEP].copy()

print(f"Cells after biological subsetting: {adata.n_obs}")

# Optional downsampling for a lighter demonstration run.
if USE_DOWNSAMPLED_EXAMPLE and adata.n_obs > DOWNSAMPLE_N_CELLS:
    rng = np.random.default_rng(RANDOM_STATE)
    cells = rng.choice(adata.obs_names, size=DOWNSAMPLE_N_CELLS, replace=False)
    adata = adata[cells].copy()
    print(f"Downsampled to {adata.n_obs} cells for a faster example analysis.")
else:
    print("Using all selected cells.")

# 4.3 Quality control and normalization

Quality control removes low-quality cells and uninformative genes before dimensionality reduction and trajectory inference. Cells with poor library complexity or excessive mitochondrial content can distort neighbourhood graphs and bias pseudotime estimates.

Thresholds should be chosen after inspecting the distributions in your own dataset. The defaults below are reasonable for this example dataset, but they are not universal.

In [ ]:
# Basic quality-control filtering.
sc.pp.filter_cells(adata, min_genes=MIN_GENES_PER_CELL)
sc.pp.filter_genes(adata, min_cells=MIN_CELLS_PER_GENE)

# Compute per-cell QC metrics.
adata.var["mt"] = adata.var_names.str.upper().str.startswith("MT-")
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], inplace=True)

# Adjust these thresholds after inspecting your own distributions.
adata = adata[
    adata.obs.n_genes_by_counts.between(N_GENES_LOWER, N_GENES_UPPER)
    & (adata.obs.pct_counts_mt < MAX_MT_PCT)
    & (adata.obs.total_counts < MAX_TOTAL_COUNTS),
    :
].copy()

print(f"Cells after QC: {adata.n_obs}")

# Standard Scanpy preprocessing.
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, n_top_genes=N_TOP_HVGS)
sc.pp.pca(adata)

### QC checkpoint

Inspect the retained distributions after filtering. This helps confirm that the chosen thresholds removed obvious outliers without discarding an entire biological state or a rare transitional population.

In [ ]:
sc.pl.violin(
    adata,
    ["total_counts", "n_genes_by_counts", "pct_counts_mt"],
    jitter=0.4,
    multi_panel=True,
)

adata.obs[CELLTYPE_KEY].value_counts()

# 4.4 Batch and patient-effect handling, clustering, and UMAP visualization

This is one of the most important parts of the workflow. Many trajectory methods depend directly or indirectly on the latent representation and neighbourhood graph used as input. Overcorrection can collapse biologically meaningful states, whereas insufficient correction can make trajectories follow patient identity instead of cellular progression.

The goal is to preserve biologically plausible continuity while minimizing unwanted patient-specific structure.

In [ ]:
# Build an initial neighbourhood graph and embedding before batch correction.
sc.pp.neighbors(adata, n_neighbors=N_NEIGHBORS, n_pcs=min(N_PCS, adata.obsm["X_pca"].shape[1]))
sc.tl.umap(adata, min_dist=UMAP_MIN_DIST)
sc.tl.leiden(adata, resolution=LEIDEN_RESOLUTION, flavor="igraph", n_iterations=2)

In [ ]:
sc.pl.umap(adata, color=[CELLTYPE_KEY, PATIENT_KEY])

## 4.4.1 Batch correction rationale

Because cells originate from multiple patients, patient-specific effects may dominate the low-dimensional structure if they are not addressed. Here we use Harmony to mitigate patient effects in PCA space before rebuilding the neighbour graph and UMAP.

Alternative integration methods such as scVI can also be appropriate. Regardless of the method, always check that known T-cell states remain distinguishable and that rare transitional populations are not lost.

In [ ]:
sc.pp.pca(adata)
pcs = adata.obsm["X_pca"]

# Run Harmony on PCA coordinates to reduce patient-specific effects.
harmony_out = hm.run_harmony(pcs, adata.obs, PATIENT_KEY)
adata.obsm["X_pca_harmony"] = harmony_out.Z_corr

# Recompute the graph and embedding using the corrected representation.
sc.pp.neighbors(
    adata,
    use_rep="X_pca_harmony",
    n_neighbors=N_NEIGHBORS,
    n_pcs=min(N_PCS, adata.obsm["X_pca_harmony"].shape[1]),
)
sc.tl.umap(adata, min_dist=UMAP_MIN_DIST)
sc.tl.leiden(adata, resolution=LEIDEN_RESOLUTION, flavor="igraph", n_iterations=2)

**Interpretation checkpoint.** After correction, confirm that cells cluster primarily by biology rather than by patient, that rare or transitional states are still visible, and that the global UMAP continuity is biologically plausible. Only move to trajectory inference once this looks reasonable.

In [ ]:
sc.pl.umap(adata, color=[CELLTYPE_KEY, PATIENT_KEY])

# 4.5 Trajectory inference analysis

Trajectory inference reconstructs putative cell-state progression from transcriptomic snapshots. Because no single method is perfect, this notebook uses three complementary approaches:

- **PAGA** to evaluate coarse-grained connectivity between annotated states
- **Monocle 3** to learn a principal graph and assign pseudotime values
- **Slingshot** to provide an independent robustness check based on cluster relationships

Using more than one method helps separate stable biological structure from method-specific artifacts.

## 4.5.1 Root selection criteria

Choosing the root is a biological decision, not just a visual one. In T-cell analyses, roots are often assigned to naive-like or less differentiated states on the basis of:

- prior biological knowledge
- canonical marker genes
- expected differentiation hierarchies
- consistency between annotation and topology

In this example, we use a naive-like state as the root and support that choice with marker expression.

In [ ]:
sc.tl.rank_genes_groups(adata, groupby=CELLTYPE_KEY, method="wilcoxon")

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata, n_genes=3)

Use the marker-gene overview to verify that the annotation is biologically sensible before committing to a root. For instance, a naive-like cluster should show markers consistent with a less differentiated T-cell state, while Treg-, Tfh-, or Th17-like states should retain their expected signatures.

Root choice directly affects pseudotime orientation, so this checkpoint is worth treating seriously.

## 4.5.2 PAGA analysis

PAGA is used first to evaluate whether the selected populations form a connected manifold and whether the inferred branch structure is plausible. It is especially helpful for identifying disconnected states that should perhaps be excluded from the trajectory analysis before finer inference begins.

In [ ]:
trajectory_rep = "X_pca_harmony" if "X_pca_harmony" in adata.obsm else "X_pca"
root_celltype = ROOT_CELLTYPE

sc.pp.neighbors(adata, use_rep=trajectory_rep, n_neighbors=N_NEIGHBORS)
sc.tl.diffmap(adata)
sc.pp.neighbors(adata, use_rep="X_diffmap")
sc.tl.paga(adata, groups=CELLTYPE_KEY)

iroot = np.flatnonzero(adata.obs[CELLTYPE_KEY] == root_celltype)[0]
adata.uns["iroot"] = iroot
sc.tl.dpt(adata)

In [ ]:
sc.pl.paga(adata, color=[CELLTYPE_KEY])
sc.pl.umap(adata, color=[CELLTYPE_KEY, "dpt_pseudotime"], legend_loc="on data")

## 4.5.3 Monocle 3 analysis

Monocle 3 learns a principal graph in reduced-dimensional space and provides a cell-level pseudotime ordering. The exact graph can be sensitive to the input embedding and root assignment, so parameter tuning should be revisited if the output looks biologically implausible.

The `MONOCLE_ROOT_INDEX` parameter below is an example setting for this notebook. In a new dataset, choose a root that matches the biologically expected starting state.

In [1]:
projected_points, mst, centroids = learn_graph(
    matrix=adata.obsm["X_umap"],
    clusters=adata.obs[CELLTYPE_KEY],
)
pseudotime = order_cells(
    adata.obsm["X_umap"],
    centroids,
    mst=mst,
    projected_points=projected_points,
    root_cells=MONOCLE_ROOT_INDEX,
)
adata.obs["monocle3_pseudotime"] = pseudotime

umap = adata.obsm["X_umap"]
celltypes = pd.Index(adata.obs[CELLTYPE_KEY].unique())
palette = dict(zip(celltypes, plt.cm.tab10(np.linspace(0, 1, len(celltypes)))))
color = adata.obs[CELLTYPE_KEY].map(palette)

NameError: name 'learn_graph' is not defined

In [ ]:
plt.figure(1, (8, 6))
plt.title("Principal graph")
plt.scatter(umap[:, 0], umap[:, 1], c=color, s=1, cmap="nipy_spectral")

edges = np.array(mst.nonzero()).T
for edge in edges:
    plt.plot(centroids[edge, 0], centroids[edge, 1], c="black", linewidth=1)
plt.xticks([])
plt.yticks([])
plt.show()

Visualize the resulting pseudotime on the same UMAP. A sensible ordering should agree with the PAGA topology, the chosen root population, and the expected biological progression of the selected T-cell states.

In [ ]:
sc.pl.umap(
    adata,
    color=[CELLTYPE_KEY, "monocle3_pseudotime"],
    cmap="viridis",
    legend_loc="on data",
)

In [ ]:
## 4.5.4 Slingshot analysis

Slingshot provides an independent view of lineage structure by fitting smooth curves through cluster relationships. In this protocol, it serves mainly as a robustness check: if the same broad progression appears with Monocle 3 and Slingshot, confidence in the trajectory interpretation increases.

fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(10, 10))
custom_xlim = (-12, 12)
custom_ylim = (-12, 12)

In [ ]:
slingshot = Slingshot(
    adata,
    celltype_key=CELLTYPE_KEY,
    obsm_key="X_umap",
    start_node=SLINGSHOT_START_NODE,
    is_debugging="verbose",
)
slingshot.fit(num_epochs=1, debug_axes=axes)

fig, axes = plt.subplots(ncols=2, figsize=(12, 4))
axes[0].set_title("Clusters")
axes[1].set_title("Pseudotime")
slingshot.plotter.curves(axes[0], slingshot.curves)
slingshot.plotter.clusters(axes[0], labels=np.arange(slingshot.num_clusters), s=4, alpha=0.5)
slingshot.plotter.clusters(axes[1], color_mode="pseudotime", s=5)

for ax in axes:
    ax.set_xlim(custom_xlim)
    ax.set_ylim(custom_ylim)

## 4.5.5 Trajectory validation

Trajectory validation should combine computational consistency and biological plausibility:

- **PAGA** validates the global topology
- **Monocle 3** provides the principal graph and pseudotime ordering
- **Slingshot** tests robustness to the inference framework

In parallel, inspect marker trends, branch structure, and the agreement between the inferred trajectory and published T-cell biology. Strong claims should not rely on a single embedding or a single method.

A useful biological validation step is to check whether the inferred CD4 trajectory agrees with published evidence for CD4 T-cell plasticity. In this example, a trajectory that starts from a naive-like state and branches toward Treg-, Tfh-, or Th17-like programs should be supported both by marker expression and by the literature.

Example reference:
<https://www.cell.com/immunity/fulltext/S1074-7613(09)00198-8?_returnURL=https%3A%2F%2Flinkinghub.elsevier.com%2Fretrieve%2Fpii%2FS1074761309001988%3Fshowall%3Dtrue>

# 4.6 Transcription factor activity analysis along pseudotime

This section uses `decoupler` together with the CollecTRI TF-target resource to estimate transcription factor activity scores and rank TF programs that vary along pseudotime.

This is not a classical count-based differential expression test. Instead, it is a regulatory interpretation layer that helps prioritize candidate drivers before moving to explicit gene regulatory network inference.

In [ ]:
# Load a curated TF-target network for activity scoring.
collectri = dc.op.collectri(organism="human")

In [ ]:
dc.mt.ulm(data=adata, net=collectri)
score = dc.pp.get_obsm(adata=adata, key="score_ulm")

In [ ]:
tfs = dc.tl.rankby_order(adata=score, order="dpt_pseudotime", stat="dcor")
tfs.head(20)

In [ ]:
top_tfs = tfs.head(10)["name"].to_list()
sc.pl.umap(
    score,
    color=top_tfs,
    cmap="coolwarm",
    ncols=5,
    title=[f"{t} score" for t in top_tfs],
)

In [ ]:
bin_tfs = dc.pp.bin_order(
    adata=score,
    order="dpt_pseudotime",
    names=top_tfs,
    label=CELLTYPE_KEY,
)
dc.pl.order(df=bin_tfs, mode="line", figsize=(6, 3))

# 4.7 Genes varying along the principal graph

Next, we identify genes whose expression changes smoothly along the learned graph. This helps highlight branch-associated programs that might not be obvious from cluster-level markers alone.

In [ ]:
expression_matrix = sparse.csr_matrix(adata.X)

This graph-aware statistic ranks genes by how strongly their expression varies along the principal graph learned above. The top genes can be used for branch interpretation, candidate marker discovery, and downstream pathway analysis.

In [ ]:
morans_i_scores, pvalues, adjusted_pvalues = differential_expression_genes(
    expression_matrix,
    projected_cells=projected_points,
)
morani_results = pd.DataFrame(
    {
        "Moran's I score": morans_i_scores,
        "P-values": np.round(pvalues, 4),
        "Adjusted P-values": np.round(adjusted_pvalues, 4),
        "Gene Indices": np.arange(len(morans_i_scores)),
    },
    index=pd.Series(adata.var_names, name="Genes"),
).sort_values("Moran's I score", ascending=False)

morani_results.head(20)

In [ ]:
plt.figure(12, (20, 10))
for i, gene_idx in enumerate(morani_results["Gene Indices"].values[:12]):
    plt.subplot(3, 4, i + 1)
    plt.scatter(umap[:, 0], umap[:, 1], c=expression_matrix[:, gene_idx].toarray(), s=3, cmap="YlOrRd")
    plt.xticks([])
    plt.yticks([])
    plt.xlabel(adata.var_names[gene_idx])
plt.show()

In [ ]:
# 4.8 Pathway enrichment analysis

Pathway analysis provides a higher-level biological summary of the signatures observed along pseudotime or across branches. In this notebook, we first score Hallmark gene sets directly inside Python and then note an optional Reactome workflow for web-based enrichment.

hallmark = dc.op.hallmark(organism="human")
dc.mt.ulm(data=adata, net=hallmark)
score = dc.pp.get_obsm(adata=adata, key="score_ulm")

In [ ]:
sc.pl.umap(score, color=["MTORC1_SIGNALING", CELLTYPE_KEY], cmap="RdBu_r", vcenter=0)
sc.pl.violin(score, keys=["MTORC1_SIGNALING"], groupby=CELLTYPE_KEY, rotation=90)

In [ ]:
pws = dc.tl.rankby_order(
    adata=score,
    order="dpt_pseudotime",
)

In [ ]:
bin_pws = dc.pp.bin_order(
    adata=score,
    order="dpt_pseudotime",
    names=pws.head(5)["name"].to_list(),
    label=CELLTYPE_KEY,
)
bin_pws

In [ ]:
dc.pl.order(
    df=bin_pws,
    mode="mat",
    kw_order={"vmin": -1, "vmax": +10, "cmap": "RdBu_r"},
    figsize=(6, 3),
)

In [ ]:
## Optional web-based Reactome enrichment with Enrichr

If you want a complementary Reactome analysis, export the branch-associated genes or top pseudotime-associated genes and submit them to Enrichr:

<https://maayanlab.cloud/Enrichr/>

This is a convenient way to validate whether the same biological themes detected with Hallmark scoring are also recovered in a curated pathway database.

# 4.9 Gene regulatory network inference with pySCENIC

This section marks the transition from the more beginner-friendly part of the notebook to a more advanced workflow. It assumes that pySCENIC has already been run outside the notebook and that the required adjacency and regulon files are available locally.

## What this section expects
- a pySCENIC adjacency table
- regulon annotations
- enough memory to run pySCENIC outside the notebook, typically on an HPC system

## Why this is separated
The computationally intensive pySCENIC steps are best executed in a reproducible command-line or containerized environment. The notebook below focuses on importing the resulting network, scoring TF activity, and interpreting candidate regulators in the context of the inferred trajectory.

In [ ]:
# Replace these paths with the location of your own pySCENIC output files.
SCENIC_DIR = "/path/to/pyscenic/output"
SCENIC_ADJACENCIES = f"{SCENIC_DIR}/expr_mat.adjacencies.tsv"
SCENIC_REGULONS = f"{SCENIC_DIR}/regulons.csv"
TOP_EDGE_FRACTION = 0.10

grn = (
    pd.read_csv(SCENIC_ADJACENCIES, index_col=False, sep="\t")
    .rename(columns={"TF": "source", "importance": "score"})
)

top_n = int(np.ceil(grn.shape[0] * TOP_EDGE_FRACTION))
grn = grn.sort_values("score", ascending=False).head(top_n)

cols = ["source", "target", "score"]

# Keep only target relationships retained after regulon pruning.
reg = pd.read_csv(SCENIC_REGULONS)
reg = reg.iloc[2:, [0, 8]]
reg.columns = ["source", "target"]
reg["target"] = reg["target"].str.split(",")
reg_exp = reg.explode("target")
reg_exp["target"] = reg_exp["target"].str.replace(r"[\[\(\)' ]", "", regex=True)
grn = pd.merge(grn, reg_exp, on=["source", "target"], how="inner")

grn = grn[cols]
grn = grn.sort_values(["source", "target"])
grn = grn.drop_duplicates(subset=["source", "target"])

In [ ]:
dc.mt.ulm(data=adata, net=grn)
score = dc.pp.get_obsm(adata=adata, key="score_ulm")

The next plots rank regulons by their association with pseudotime. This helps prioritize TF programs that change gradually along the inferred trajectory before focusing on more specific branch-level comparisons.

In [ ]:
tfs = dc.tl.rankby_order(adata=score, order="dpt_pseudotime", stat="dcor",)
#tfs

In [ ]:
top_tfs = tfs.head(10)["name"].to_list()
sc.pl.umap(score, color=top_tfs, cmap="coolwarm", ncols=5, title=[f"{t} score" for t in top_tfs]) #vmin=-5, vmax=5, 

In [ ]:
bin_tfs = dc.pp.bin_order(
    adata=score,
    order="dpt_pseudotime",
    names=top_tfs,
    label=CELLTYPE_KEY,
)
dc.pl.order(df=bin_tfs, mode="line", figsize=(6, 3))

In [ ]:
# Convert to DataFrame
df = adata.to_df()  # cells × genes
df['celltype'] = adata.obs['celltype'].values
mean_expr = df.groupby('celltype').mean()


In [ ]:
df_naive = mean_expr.loc[['Naive']]

In [ ]:
tf_acts, tf_padj = dc.mt.ulm(data=df_naive, net=grn)

# Filter by sign padj
msk = (tf_padj.T < 0.05).iloc[:, 0]
tf_acts = tf_acts.loc[:, msk]

In [ ]:
tf_acts.max().sort_values()

In [ ]:
dc.pl.network(
    net=grn,
    data=df_naive,
    score=tf_acts,
    sources=["FOXP3"],
    targets=5,
    figsize=(5, 5),
    vcenter=True,
    by_abs=True,
    size_node=10,
)

## 4.10 Differential regulon activity across branches

Once a GRN has been imported, you can compare regulon activity between specific branches or endpoints. This is especially useful for prioritizing candidate regulators that might underlie divergence between alternative T-cell fates.

The example below focuses on the `Naive -> Treg` branch, but the same logic can be applied to any lineage pair of interest.

In [ ]:
adata_new = adata[adata.obs[CELLTYPE_KEY].isin(["Naive", "Treg"])].copy()

In [ ]:
dc.mt.ulm(data=adata_new, net=grn)
score = dc.pp.get_obsm(adata=adata_new, key="score_ulm")

In [ ]:
tfs = dc.tl.rankby_order(adata=score, order="dpt_pseudotime", stat="dcor",)

In [ ]:
top_tfs = tfs.head(5)["name"].to_list()
sc.pl.umap(score, color=top_tfs, cmap="coolwarm", ncols=5, title=[f"{t} score" for t in top_tfs]) #vmin=-5, vmax=5, 

In [ ]:
sc.pl.umap(adata_new, color=CELLTYPE_KEY)

In [ ]:
bin_tfs = dc.pp.bin_order(
    adata=score,
    order="dpt_pseudotime",
    names=top_tfs,
    label=CELLTYPE_KEY,
)
dc.pl.order(df=bin_tfs, mode="line", figsize=(6, 3))

## 4.11 TF-TF network construction and interpretation

A compact TF-TF network view can make the GRN easier to interpret. Here we filter the imported network to a curated TF list (the Lambert TF compendium) and then visualize example subnetworks for selected cell states.

This section is intentionally more advanced. Treat it as a template for custom follow-up analyses rather than a one-size-fits-all endpoint.

In [ ]:
# Replace this path with the location of your preferred TF reference list.
LAMBERT_TF_LIST = "/path/to/lambert_tf_list.txt"

lambert = pd.read_csv(LAMBERT_TF_LIST, sep="\t", header=None)[0].tolist()
tf_grn = grn[grn["source"].isin(lambert) & grn["target"].isin(lambert)].copy()

In [ ]:
# Summarize example TF-TF activity in two cell states of interest.
df = adata.to_df()
df["celltype"] = adata.obs[CELLTYPE_KEY].values
mean_expr = df.groupby("celltype").median()

df_naive = mean_expr.loc[["Naive"]]
tf_acts_n, tf_padj = dc.mt.ulm(data=df_naive, net=tf_grn)

df_treg = mean_expr.loc[["Treg"]]
tf_acts_t, tf_padj = dc.mt.ulm(data=df_treg, net=tf_grn)

In [ ]:
dc.pl.network(
    net=tf_grn,
    data=df_naive,
    score=tf_acts_n,
    sources=["IKZF2", "BACH2","FOXP3", "KLF2", "KLF3"],
    targets=5,
    figsize=(5, 5),
    vcenter=True,
    by_abs=True,
    size_node=10,
)

In [ ]:
dc.pl.network(
    net=tf_grn,
    data=df_treg,
    score=tf_acts_t,
    sources=["IKZF2", "BACH2", "FOXP3", "KLF2", "KLF3"],
    targets=10,
    figsize=(5, 5),
    vcenter=True,
    by_abs=True,
    size_node=10,
)